# Introduction
This project analyzes Teach for America applicant data to predict whether an applicant will complete the admissions process. Teach for America is a program that aims to recruit teachers in the United States. Its business problem is that applicants' drop-off rate is increasing, creating inefficiencies in recruitment operations. Identifying applicants at risk of not completing the process could help the recruitment team prioritize outreach and provide targeted support.

By predicting which applicants are likely to complete the process, Teacher America can focus its resources on reducing the drop-off rate and improving the overall quality of its teacher recruitment efforts. We will compare Random Forest and Decision Trees, and model performance will be evaluated using accuracy, Kappa, and sensitivity metrics.

The dataset includes application dates, recruitment interactions, academic information, and essay characteristics. You can find this dataset, the People Analytics at Teach for America (A) case (# 418013 and # 418715), at https://www.iveypublishing.ca/.

## Import libraries and load dataset

In [ ]:
packages <- c("readxl", "caret", "C50", "ranger", "recipes")

for (pkg in packages) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    install.packages(pkg)
  }
}

invisible(lapply(packages, library, character.only = TRUE))

# Load data relative to the notebook directory
df_data <- as.data.frame(readxl::read_excel(
  "../labs/teacher_america.xlsx",
  sheet = "Data for HBS-TFA Case - 2018-02"
))

# Dataset Description
There are 74,839 rows and 39 columns in the dataset without any missing values. The target variable is "Completed Admissions Process", which is a binary variable indicating whether an applicant completed the admissions process (1) or not (0). The dataset contains various features related to the applicants' application process, school information, and essay information.
Major, minor, and university information also have been cleaned and categorized into broader categories for easier analysis, and we will use these for our predictive models instead of the original variables because of its computational efficiency.

Only 81% of applicants completed the admissions process, while 21% did not. This indicates that the dataset is imbalanced, which may affect the performance of the predictive models. We will need to consider this issue during model training and evaluation.

In [ ]:
# Capture original columns immediately after loading the dataset
original_columns <- names(df_data)

# Calculate the mode, excluding missing values
get_mode <- function(x) {
  x <- x[!is.na(x)]
  if (!length(x)) return(NA_character_)

  values <- unique(x)
  counts <- tabulate(match(x, values))
  paste(as.character(values[counts == max(counts)]), collapse = ", ")
}

# Summarize original columns
structure_table <- do.call(rbind, lapply(original_columns, function(column) {
  x <- df_data[[column]]
  numeric_column <- is.numeric(x) && any(!is.na(x))

  data.frame(
    Variable = column,
    Type = paste(class(x), collapse = "/"),
    Missing = sum(is.na(x)),
    Unique = length(unique(x[!is.na(x)])),
    Mean = if (numeric_column) mean(x, na.rm = TRUE) else NA_real_,
    Median = if (numeric_column) median(x, na.rm = TRUE) else NA_real_
  )
}))

# Display the summary table
IRdisplay::display_html(as.character(knitr::kable(
  structure_table,
  format = "html",
  digits = 2,
  caption = "Original Dataset Summary",
  escape = TRUE
)))

Variable,Type,Missing,Unique,Mean,Median
Person Id,numeric,0,74839,37433.13,37431.00
App Started Year (RTAT),numeric,0,2,2015.52,2016.00
Cumulative GPA,numeric,0,225,3.17,3.23
"Is Math, Sci, or Eng Major Minor",numeric,0,2,0.16,0.00
Major 1,character,0,171,NA,NA
Major 2,character,0,171,NA,NA
Minor,character,0,171,NA,NA
Undergraduate University,character,0,1991,NA,NA
School Selectivity,character,0,6,NA,NA
Major 1 - Cleaned,character,0,26,NA,NA


# Dataset Preprocessing and Cleaning
We need to factorize some numeric variables such as application year, event attended, met, invited, responded, region preference level, school selectivity, major1, major2, minor, university, stem, and completed admissions process.

In [19]:
df_data$admission_completed <- factor(df_data$`Completed Admissions Process`)
df_data$application_year <- factor(df_data$`App Started Year (RTAT)`)
df_data$event_attended <- factor(df_data$`Attended Event`)
df_data$met <- factor(df_data$Met)
df_data$invited <- factor(df_data$Invited)
df_data$responded <- factor(df_data$Responded)
df_data$region <- factor(df_data$`Region Preference Level`)
df_data$school_selectivity <- factor(df_data$`School Selectivity`)
df_data$major1 <- factor(df_data$`Major 1 - Cleaned`)
df_data$major2 <- factor(df_data$`Major 2 - Cleaned`)
df_data$minor <- factor(df_data$`Minor - Cleaned`)
df_data$university <- factor(df_data$`Undergraduate University - Cleaned`)
df_data$stem <- factor(df_data$`Is Math, Sci, or Eng Major Minor`)


Then we create new variables for application duration gaps and whether the application deadline was met or not to reduce the number of categorical variables and computational costs. The new variables can give more information than the date columns alone.

In [20]:
df_data$Application.Deadline <- as.Date(df_data$`Application Deadline`, format = "%Y-%m-%d")
df_data$Submitted.Date <- as.Date(df_data$`Submitted Date`, format = "%Y-%m-%d")
df_data$Started.Date <- as.Date(df_data$`Started Date`, format = "%Y-%m-%d")
df_data$Sign.up.Date <- as.Date(df_data$`Sign-up Date`, format = "%Y-%m-%d")

#creating duration gap between sign-up date & start apply date
df_data$signUpProcessToStartApply <- df_data$Started.Date - df_data$Sign.up.Date

#creating duration gap between start apply date & submit date
df_data$startApplyToSubmission <- df_data$Submitted.Date - df_data$Started.Date
#creating whether deadline meet or not
df_data$deadline_meet <- factor(df_data$Submitted.Date <= df_data$Application.Deadline)

We will change the name of the variables for easier reference in the code, then use the new variable names to create new data frame later.

In [21]:
df_data$gpa <- df_data$`Cumulative GPA`
df_data$essay1 <- df_data$`Essay 1 Length`
df_data$essay2 <- df_data$`Essay 2 Length`
df_data$essay3 <- df_data$`Essay 3 Length`
df_data$essay_words <- df_data$`Essays Unique Words`
df_data$topic0 <- df_data$`Essays Topic 0`
df_data$topic1 <- df_data$`Essays Topic 1`
df_data$topic2 <- df_data$`Essays Topic 2`
df_data$topic3 <- df_data$`Essays Topic 3`
df_data$topic4 <- df_data$`Essays Topic 4`
df_data$topic5 <- df_data$`Essays Topic 5`
df_data$topic6 <- df_data$`Essays Topic 6`
df_data$topic7 <- df_data$`Essays Topic 7`
df_data$topic8 <- df_data$`Essays Topic 8`
df_data$topic9 <- df_data$`Essays Topic 9`
df_data$essay_sentiment <- df_data$`Essays Sentiment`

After data cleaining, you can use 32 variables, excluding Confirmed TFA Offer as it is known after the admission process and cause data leakage in our models.

Then we create 80% of the data for training and 20% for testing. As we explored above, the dataset is skewed, so we will use stratified sampling to ensure that the distribution of the target variable is similar in both sets.

In [ ]:

selected_all <- c("admission_completed"
                  , "event_attended", "met", "invited", "responded", "region"
                  , "application_year",  "deadline_meet", "signUpProcessToStartApply", "startApplyToSubmission"
                  , "school_selectivity", "major1", "major2", "minor", "university", "stem", "gpa"
                  , "essay1","essay2","essay3","essay_words"
                  , "topic0", "topic1","topic2", "topic3", "topic4","topic5", "topic6", "topic7","topic8","topic9"
                  ,"essay_sentiment"
                  )

df_all <- df_data[,selected_all]
idx <- createDataPartition(df_all$admission_completed, p=0.8, list=FALSE)

train_set <- df_all[idx,]
test_set <- df_all[-idx,]


# Summarize columns in df_all
structure_table <- do.call(rbind, lapply(names(df_all), function(column) {
  x <- df_all[[column]]
  numeric_column <- is.numeric(x) && any(!is.na(x))

  data.frame(
    Variable = column,
    Type = paste(class(x), collapse = "/"),
    Missing = sum(is.na(x)),
    Unique = length(unique(x[!is.na(x)])),
    Mean = if (numeric_column) mean(x, na.rm = TRUE) else NA_real_,
    Median = if (numeric_column) median(x, na.rm = TRUE) else NA_real_
  )
}))

# Display the summary table
IRdisplay::display_html(as.character(knitr::kable(
  structure_table,
  format = "html",
  digits = 2,
  caption = "Modeling Dataset Summary",
  escape = TRUE
)))


Variable,Type,Missing,Unique,Mean,Median
admission_completed,factor,0,2,NA,NA
event_attended,factor,0,2,NA,NA
met,factor,0,2,NA,NA
invited,factor,0,2,NA,NA
responded,factor,0,2,NA,NA
region,factor,0,3,NA,NA
application_year,factor,0,2,NA,NA
deadline_meet,factor,0,2,NA,NA
signUpProcessToStartApply,difftime,0,352,NA,NA
startApplyToSubmission,difftime,0,229,NA,NA


# Train models - Decision Tree & Random Forest
## Decision Tree using caret package with C5.0Cost method.

So, all preparation is set! Let's train models!

First, we start training Decision Tree models, using the C5.0 algorithm with cost-sensitive learning to handle the imbalanced dataset. We will use 10-fold cross-validation for both Decision Tree and Random Forest.

In [24]:
tune_dt <- expand.grid(
  trials = seq(5, 15, 5),
  model = c("rules", "tree"),
  winnow = c(TRUE, FALSE),
  cost = c(1, 2, 4)
)

In [ ]:
dt1 <- train(admission_completed~.
              , data=train_set
              , method="C5.0Cost"
              , metric="Kappa"
              , trControl=trainControl(method="cv",number=10)
              , tuneGrid = tune_dt
)

Let's check the confusion matrix for this model. We set the positive class as "0" because we are interested in predicting the applicants who did not complete the admissions process.

In [36]:
dt1_cm <- confusionMatrix(predict(dt1, newdata = test_set)
            , test_set$admission_completed
            , positive = "0")
dt1_cm

Confusion Matrix and Statistics

          Reference
Prediction     0     1
         0   608   960
         1  2274 11125
                                          
               Accuracy : 0.7839          
                 95% CI : (0.7772, 0.7905)
    No Information Rate : 0.8074          
    P-Value [Acc > NIR] : 1               
                                          
                  Kappa : 0.1592          
                                          
 Mcnemar's Test P-Value : <2e-16          
                                          
            Sensitivity : 0.21096         
            Specificity : 0.92056         
         Pos Pred Value : 0.38776         
         Neg Pred Value : 0.83029         
             Prevalence : 0.19256         
         Detection Rate : 0.04062         
   Detection Prevalence : 0.10476         
      Balanced Accuracy : 0.56576         
                                          
       'Positive' Class : 0               
                  

## Random Forest using the ranger package.
We will use the same training and test sets as for the Decision Tree models to compare the performance of the models.

In [ ]:
tune_rf <- expand.grid(
  mtry = c(3, 5, 10, 20),
  splitrule = "gini",
  min.node.size = c(1, 5, 10)
)

rf1 <- train(
  admission_completed ~ .,
  data = train_set,
  method = "ranger",
  metric = "Kappa",
  trControl = trainControl(method = "cv", number = 10),
  tuneGrid = tune_rf,
  num.trees = 100,
  importance = "permutation",
  respect.unordered.factors = "order",
  num.threads = 1
)

In [35]:
rf1_cm <- confusionMatrix(predict(rf1, newdata = test_set)
            , test_set$admission_completed
            , positive = "0")

rf1_cm

Confusion Matrix and Statistics

          Reference
Prediction     0     1
         0    33    61
         1  2849 12024
                                          
               Accuracy : 0.8056          
                 95% CI : (0.7991, 0.8119)
    No Information Rate : 0.8074          
    P-Value [Acc > NIR] : 0.7231          
                                          
                  Kappa : 0.0101          
                                          
 Mcnemar's Test P-Value : <2e-16          
                                          
            Sensitivity : 0.011450        
            Specificity : 0.994952        
         Pos Pred Value : 0.351064        
         Neg Pred Value : 0.808445        
             Prevalence : 0.192557        
         Detection Rate : 0.002205        
   Detection Prevalence : 0.006280        
      Balanced Accuracy : 0.503201        
                                          
       'Positive' Class : 0               
                  

# Model Evaluation
Finally, we will evaluate the performance of two models using accuracy, Kappa, and sensitivity metrics. Although the Random Forest model achieves higher accuracy, its recall (true positive rate) is very low, meaning it misses most applicants who drop out. Therefore, the Decision Tree is the more suitable of the two models, as it has higher Kappa, precision, and recall.

In [41]:
# Store confusion matrices for each model
cm_list <- c(DecisionTree = list(dt1_cm), RandomForest = list(rf1_cm))

# Create a performance comparison table (positive class: 0)
results <- do.call(rbind, lapply(names(cm_list), function(name) {
  cm <- cm_list[[name]]
  data.frame(
    Model = name,
    Accuracy = unname(cm$overall["Accuracy"]),
    Kappa = unname(cm$overall["Kappa"]),
    Precision = unname(cm$byClass["Pos Pred Value"]),
    Recall = unname(cm$byClass["Sensitivity"]),
    F1 = unname(cm$byClass["F1"])
  )
}))

print(results, row.names = FALSE)

        Model  Accuracy      Kappa Precision     Recall         F1
 DecisionTree 0.7839246 0.15915727 0.3877551 0.21096461 0.27325843
 RandomForest 0.8055723 0.01013651 0.3510638 0.01145038 0.02217742


Additionally, we can compare the top five predictors ranked by importance in each model. This comparison can help us understand how the models differ and highlight factors the recruitment team may want to examine more closely.

However, there are two important points to keep in mind:
1. Variable importance does not imply causation.
2. Importance scores are calculated differently across models, so their absolute values are not directly comparable.

For example, GPA is the most important predictor in the Decision Tree model. However, this does not necessarily mean that applicants with higher GPAs are more likely to complete the admissions process. Second, importance scores are normalized within each model, so their absolute values should not be compared across models. Instead, let's just focus on the variable rankings.

In [40]:
# Extract the top 10 variables and importance scores
top_variables <- function(model, n = 10) {
  importance <- caret::varImp(model)$importance
  importance <- importance[
    order(importance$Overall, decreasing = TRUE),
    , drop = FALSE
  ]

  head(data.frame(
    Variable = rownames(importance),
    Score = importance$Overall
  ), n)
}

dt_top <- top_variables(dt1)
rf_top <- top_variables(rf1)

# Compare variable rankings and importance scores
importance_table <- data.frame(
  `Decision Tree` = dt_top$Variable,
  `DT Score` = dt_top$Score,
  `Random Forest` = rf_top$Variable,
  `RF Score` = rf_top$Score,
  check.names = FALSE
)

# Display rankings as row numbers
IRdisplay::display_html(as.character(knitr::kable(
  importance_table,
  format = "html",
  digits = 2,
  caption = "Top 10 Variables by Importance",
  row.names = TRUE
)))

,Decision Tree,DT Score,Random Forest,RF Score
1,gpa,100.00,essay_words,100.00
2,essay2,99.94,essay2,81.15
3,essay1,99.83,essay1,80.25
4,startApplyToSubmission,98.79,gpa,52.51
5,topic9,87.42,essay3,41.00
6,essay_words,86.85,startApplyToSubmission,27.20
7,minormanagement,78.24,topic6,16.42
8,region2,72.12,topic2,10.44
9,topic6,69.01,topic9,8.38
10,topic3,68.60,essay_sentiment,8.03


# Conclusion
In this final section, we will investigate ways that TFA recruiment teams can make use of the predictions.

Although the Decision Tree performed better, the Random Forest’s near-zero Kappa indicates little agreement beyond chance, limiting its usefulness for prediction in its current form. Nevertheless, the models share several highly ranked predictors.

Both models identified GPA, Essay 1 and Essay 2 lengths, unique essay word count, and the time between starting and submitting an application as important predictors. This overlap suggests that academic information, essay characteristics, and application timing provide useful predictive information. However, variable importance does not establish causation or indicate the direction of these relationships.

With further validation, recruiters could use model flags to offer applicants timely guidance on the next stage of the process. Such outreach should support applicants rather than restrict their opportunities.